# Pipeline 4: Counseling Session Effectiveness
## What Makes a Therapy Session Successful?

**Notebook:** `counseling-effectiveness.ipynb`  
**Domain:** Case Management — Process Recordings  
**Author:** IS 455 ML Pipelines

---

## 1. Problem Framing

### Business Problem
With 2,819 counseling sessions recorded, the organization has a rich dataset of social worker interactions. Leadership wants to know: Are our sessions actually moving the needle? What session characteristics are associated with emotional improvement? And can we flag, in advance, which sessions are likely to surface concerns — so supervisors can be on call?

**Two specific questions:**
1. **(Explanatory)** What session characteristics — duration, type, interventions applied, social worker — are most associated with positive emotional shifts in residents?
2. **(Predictive)** Can we predict whether a session will result in a concern being flagged (`concerns_flagged = True`)? This would allow proactive supervisor notification.

### Who Cares About This?
- **Social workers** — understand which techniques drive better emotional outcomes
- **Supervisors** — be alerted when a session is likely to surface a serious concern
- **Leadership** — evaluate which intervention types to invest in and standardize

### Predictive vs. Explanatory Approach
- **Explanatory (OLS Linear Regression):** Does session duration have a linear relationship with emotional improvement? Which interventions have the strongest coefficients?
- **Predictive (Gradient Boosting Classifier):** Predict `concerns_flagged` from pre-session variables (session type, duration, resident history). Deploy as a supervisor alert system.

**Data size advantage:** 2,819 sessions — the largest table. This gives the predictive model a real chance to generalize.

### Success Metrics
- **Explanatory:** R², significant coefficients on intervention types, interpretable direction
- **Predictive:** ROC-AUC, Precision/Recall (especially Recall — we want to catch flagged concerns)


## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Load data ──────────────────────────────────────────────────────────────
process   = pd.read_csv('process_recordings.csv', parse_dates=['session_date'])
residents = pd.read_csv('residents.csv')
health    = pd.read_csv('health_wellbeing_records.csv', parse_dates=['record_date'])
incidents = pd.read_csv('incident_reports.csv', parse_dates=['incident_date'])

print("Process recordings:", process.shape)
print("\nColumn overview:")
print(process.dtypes.to_string())


In [ ]:
# ── Basic exploration ──────────────────────────────────────────────────────
print("=== Session Type ===")
print(process['session_type'].value_counts())

print("\n=== Emotional States (Start) ===")
print(process['emotional_state_observed'].value_counts())

print("\n=== Interventions Applied (sample) ===")
print(process['interventions_applied'].value_counts().head(15))

print("\n=== Outcomes ===")
print("progress_noted rate:", process['progress_noted'].mean().round(3))
print("concerns_flagged rate:", process['concerns_flagged'].mean().round(3))
print("referral_made rate:", process['referral_made'].mean().round(3))

print("\n=== Session duration ===")
print(process['session_duration_minutes'].describe().round(1))


In [ ]:
# ── Engineer the emotional shift feature ──────────────────────────────────
emotion_map = {
    'Distressed': 1, 'Angry': 2, 'Sad': 3, 'Withdrawn': 4,
    'Anxious': 5,  'Calm': 6,  'Hopeful': 7, 'Happy': 8
}

process['emotion_start_num'] = process['emotional_state_observed'].map(emotion_map)
process['emotion_end_num']   = process['emotional_state_end'].map(emotion_map)
process['emotion_shift']     = process['emotion_end_num'] - process['emotion_start_num']

print("Emotional shift distribution:")
print(process['emotion_shift'].describe().round(2))
print(f"\nPositive shift (improvement): {(process['emotion_shift'] > 0).mean():.1%}")
print(f"No shift:                       {(process['emotion_shift'] == 0).mean():.1%}")
print(f"Negative shift (worsening):     {(process['emotion_shift'] < 0).mean():.1%}")


In [ ]:
# ── Intervention one-hot engineering ──────────────────────────────────────
# Each session has comma-separated interventions
interventions = ['Healing', 'Teaching', 'Legal Services', 'Caring']
for intv in interventions:
    col = 'intv_' + intv.replace(' ','_').lower()
    process[col] = process['interventions_applied'].str.contains(intv, na=False).astype(int)

# Count total interventions per session
process['n_interventions'] = process[[f'intv_{i.replace(" ","_").lower()}' for i in interventions]].sum(axis=1)

# Session timing features
process['month'] = process['session_date'].dt.month
process['day_of_week'] = process['session_date'].dt.dayofweek
process['session_type_individual'] = (process['session_type'] == 'Individual').astype(int)

print("Intervention features added.")
print(process[['intv_healing','intv_teaching','intv_legal_services','intv_caring']].mean().round(3))


In [ ]:
# ── Join resident-level features ──────────────────────────────────────────
resident_feats = residents[['resident_id','initial_risk_level','case_category',
                             'sub_cat_trafficked','sub_cat_sexual_abuse','has_special_needs']].copy()
risk_map = {'Low':1,'Medium':2,'High':3,'Critical':4}
resident_feats['initial_risk_num'] = resident_feats['initial_risk_level'].map(risk_map)

# Add resident's session count BEFORE this session (as a running history feature)
process_sorted = process.sort_values(['resident_id','session_date']).copy()
process_sorted['session_count_prior'] = process_sorted.groupby('resident_id').cumcount()

# Add recent incident count (last 90 days)
def prior_incident_count(row, incident_df):
    window_start = row['session_date'] - pd.Timedelta(days=90)
    return ((incident_df['resident_id'] == row['resident_id']) &
            (incident_df['incident_date'] >= window_start) &
            (incident_df['incident_date'] < row['session_date'])).sum()

# Approximate: total incidents per resident (faster than row-wise)
incident_counts = incidents.groupby('resident_id').size().reset_index(name='total_incidents')

process_full = process_sorted     .merge(resident_feats[['resident_id','initial_risk_num','sub_cat_trafficked',
                            'sub_cat_sexual_abuse','has_special_needs']],
           on='resident_id', how='left')     .merge(incident_counts, on='resident_id', how='left')

process_full['total_incidents'] = process_full['total_incidents'].fillna(0)
process_full[['sub_cat_trafficked','sub_cat_sexual_abuse','has_special_needs']] =     process_full[['sub_cat_trafficked','sub_cat_sexual_abuse','has_special_needs']].astype(int)

print("Full process dataset shape:", process_full.shape)
print("Missing values:")
print(process_full.isnull().sum()[process_full.isnull().sum() > 0])


In [ ]:
# ── Exploratory visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Duration vs emotion shift
axes[0,0].scatter(process_full['session_duration_minutes'],
                  process_full['emotion_shift'], alpha=0.1, color='steelblue')
axes[0,0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0,0].set_xlabel('Session Duration (minutes)')
axes[0,0].set_ylabel('Emotional Shift')
axes[0,0].set_title('Duration vs. Emotional Shift', fontsize=11)

# Emotion shift by session type
process_full.boxplot(column='emotion_shift', by='session_type', ax=axes[0,1])
axes[0,1].set_title('Emotional Shift by Session Type', fontsize=11)
axes[0,1].set_xlabel('')

# Emotion shift by intervention type
intv_shifts = {intv: process_full[process_full[f'intv_{intv.replace(" ","_").lower()}'] == 1]['emotion_shift'].mean()
               for intv in interventions}
axes[0,2].bar(list(intv_shifts.keys()), list(intv_shifts.values()), color='steelblue')
axes[0,2].set_title('Mean Emotional Shift by Intervention', fontsize=11)
axes[0,2].set_xlabel('Intervention Type')
axes[0,2].tick_params(axis='x', rotation=30)

# Concerns flagged by emotional start state
concern_by_emotion = process_full.groupby('emotional_state_observed')['concerns_flagged'].mean().sort_values(ascending=False)
axes[1,0].bar(concern_by_emotion.index, concern_by_emotion.values, color='tomato')
axes[1,0].set_title('% Concerns Flagged by Starting Emotion', fontsize=11)
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].set_ylabel('Proportion Flagged')

# Session count prior vs concerns
axes[1,1].scatter(process_full['session_count_prior'],
                  process_full['concerns_flagged'].astype(float) + np.random.normal(0,0.02,len(process_full)),
                  alpha=0.05, color='purple')
axes[1,1].set_title('Session Count vs. Concerns Flagged', fontsize=11)
axes[1,1].set_xlabel('Prior Sessions with Resident')

# Concerns by initial risk
concern_by_risk = process_full.groupby('initial_risk_num')['concerns_flagged'].mean()
axes[1,2].bar(concern_by_risk.index.map({1:'Low',2:'Med',3:'High',4:'Critical'}),
              concern_by_risk.values, color='tomato')
axes[1,2].set_title('% Concerns Flagged by Resident Risk Level', fontsize=11)

plt.suptitle('Exploratory Analysis: Counseling Session Patterns', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('p4_exploration.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Explanatory Model (OLS Regression on Emotional Shift)

In [ ]:
# ── OLS: what factors drive emotional improvement? ────────────────────────
ols_df = process_full.dropna(subset=['emotion_shift','emotion_start_num']).copy()

EXPLAIN_FEATURES4 = [
    'session_duration_minutes', 'session_type_individual', 'emotion_start_num',
    'intv_healing', 'intv_teaching', 'intv_legal_services', 'intv_caring',
    'n_interventions', 'session_count_prior', 'initial_risk_num',
    'sub_cat_trafficked', 'sub_cat_sexual_abuse', 'has_special_needs', 'total_incidents'
]

X_ols4 = ols_df[EXPLAIN_FEATURES4].fillna(ols_df[EXPLAIN_FEATURES4].median())
y_ols4 = ols_df['emotion_shift']

from sklearn.preprocessing import StandardScaler
sc4 = StandardScaler()
X_ols4_scaled = pd.DataFrame(sc4.fit_transform(X_ols4), columns=EXPLAIN_FEATURES4)
X_ols4_const = sm.add_constant(X_ols4_scaled)

ols4 = sm.OLS(y_ols4.values, X_ols4_const).fit()
print(ols4.summary())


In [ ]:
# ── Coefficient plot for emotional shift model ────────────────────────────
coef4 = pd.DataFrame({
    'coef': ols4.params.drop('const'),
    'p':    ols4.pvalues.drop('const'),
    'ci_lo': ols4.conf_int().drop('const')[0],
    'ci_hi': ols4.conf_int().drop('const')[1],
}).sort_values('coef', ascending=False)

sig4 = coef4[coef4['p'] < 0.10]

fig, ax = plt.subplots(figsize=(10, 7))
colors4 = ['steelblue' if c > 0 else 'tomato' for c in coef4['coef']]
ax.barh(coef4.index, coef4['coef'], color=colors4,
        xerr=[coef4['coef'] - coef4['ci_lo'], coef4['ci_hi'] - coef4['coef']], capsize=3)
ax.axvline(0, color='black', lw=0.8)
for i, (idx, row) in enumerate(coef4.iterrows()):
    if row['p'] < 0.10:
        ax.text(row['coef'] + 0.005, i, '✓', fontsize=10, va='center', color='green')
ax.set_title('OLS Coefficients: Predictors of Emotional Shift\n(✓ = p < 0.10, standardized)', fontsize=12)
ax.set_xlabel('Coefficient (emotional state improvement)')
plt.tight_layout()
plt.savefig('p4_ols_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nModel R²: {ols4.rsquared:.4f}")
print(f"Adjusted R²: {ols4.rsquared_adj:.4f}")
print(f"\nSignificant predictors:")
print(sig4[['coef','p']].to_string())


In [ ]:
# ── Duration sweet spot analysis ──────────────────────────────────────────
# Bin duration and plot mean emotional shift
process_full['duration_bin'] = pd.cut(process_full['session_duration_minutes'],
                                       bins=[0,45,60,75,90,120], labels=['<45','45-60','60-75','75-90','90+'])
dur_shift = process_full.groupby('duration_bin')['emotion_shift'].agg(['mean','sem','count'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(dur_shift.index.astype(str), dur_shift['mean'], yerr=dur_shift['sem'],
       capsize=5, color='steelblue', error_kw={'ecolor':'black'})
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Mean Emotional Shift by Session Duration Bin (±SE)', fontsize=13)
ax.set_xlabel('Session Duration (minutes)')
ax.set_ylabel('Mean Emotional Shift')
for i, (idx, row) in enumerate(dur_shift.iterrows()):
    ax.text(i, dur_shift['mean'].max() + 0.05, f"n={int(row['count'])}", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('p4_duration_sweetspot.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Predictive Model (Gradient Boosting Classifier — Concerns Flagged)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay, confusion_matrix

# Features known BEFORE the session ends (no leakage of outcomes)
PRED_FEATURES4 = [
    'session_duration_minutes', 'session_type_individual',
    'emotion_start_num', 'intv_healing', 'intv_teaching',
    'intv_legal_services', 'intv_caring', 'n_interventions',
    'session_count_prior', 'initial_risk_num',
    'sub_cat_trafficked', 'sub_cat_sexual_abuse', 'has_special_needs',
    'total_incidents', 'month', 'day_of_week'
]

model_df4 = process_full.dropna(subset=PRED_FEATURES4 + ['concerns_flagged']).copy()
X4 = model_df4[PRED_FEATURES4]
y4 = model_df4['concerns_flagged'].astype(int)

X_tr4, X_te4, y_tr4, y_te4 = train_test_split(X4, y4, test_size=0.2, stratify=y4, random_state=42)

print(f"Train: {len(X_tr4)} | Test: {len(X_te4)}")
print(f"Concerns flagged rate: {y4.mean():.1%}")

# Compare models
models4 = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=500))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()), ('m', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'Gradient Boosting':   Pipeline([('sc', StandardScaler()), ('m', GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42))]),
}

skf4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, pipe in models4.items():
    cv = cross_validate(pipe, X_tr4, y_tr4, cv=skf4, scoring=['roc_auc','f1'])
    print(f"{name:22s}  AUC={cv['test_roc_auc'].mean():.4f}  F1={cv['test_f1'].mean():.4f}")


In [ ]:
# ── Final model evaluation ────────────────────────────────────────────────
best_pipe4 = models4['Gradient Boosting']
best_pipe4.fit(X_tr4, y_tr4)

y_proba4 = best_pipe4.predict_proba(X_te4)[:, 1]
y_hat4   = (y_proba4 >= 0.35).astype(int)  # lower threshold — prioritize recall

auc4 = roc_auc_score(y_te4, y_proba4)
print(f"Test ROC-AUC: {auc4:.4f}")
print("\nClassification Report (threshold=0.35):")
print(classification_report(y_te4, y_hat4, target_names=['No Concern','Concern Flagged']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
RocCurveDisplay.from_predictions(y_te4, y_proba4, ax=axes[0], name='Gradient Boosting')
axes[0].set_title('ROC Curve — Concerns Flagged Prediction', fontsize=12)

cm4 = confusion_matrix(y_te4, y_hat4)
sns.heatmap(cm4, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Predicted No','Predicted Concern'],
            yticklabels=['Actual No','Actual Concern'])
axes[1].set_title('Confusion Matrix (threshold=0.35)', fontsize=12)
plt.tight_layout()
plt.savefig('p4_roc_confusion.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
gb4 = best_pipe4.named_steps['m']
fi4 = pd.DataFrame({
    'feature': PRED_FEATURES4,
    'importance': gb4.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi4['feature'][::-1], fi4['importance'][::-1], color='darkorange')
ax.set_title('Feature Importances — Concerns Flagged Classifier', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('p4_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top features:")
print(fi4.head(10).to_string(index=False))


## 6. Evaluation & Interpretation

### Business Interpretation

**What OLS tells us (Explanatory):**
- The emotional shift model reveals which session characteristics are associated with residents *feeling better* by the end.
- If `intv_healing` has a positive, significant coefficient, it suggests Healing-focused sessions may be more effective at improving emotional state, all else equal.
- The duration sweet-spot analysis reveals whether longer sessions have diminishing returns.

**What Gradient Boosting tells us (Predictive):**
- Given a session's setup (type, duration, resident history, planned interventions), the model estimates the probability that a concern will be flagged.
- This can be surfaced as a supervisor alert: "Session with Resident #12 today has a 68% probability of surfacing a concern — supervisor should be available."

**Consequences of errors:**
- **False negative (missing a concern):** Supervisor not available when a session surfaces a serious issue. High cost.
- **False positive (flagging a normal session):** Supervisor on standby unnecessarily. Low cost.
- **We set threshold at 0.35** to prioritize recall (catch more true concerns at cost of some false alarms).

## 7. Causal & Relationship Analysis

**Key relationships found:**
- Starting emotional state is strongly predictive of both shift and concerns — lower starting states are associated with more concerns, which is theoretically expected.
- The number of prior sessions shows a pattern — residents earlier in their time in care may be more volatile.
- Intervention type associations with emotional shift require careful interpretation: social workers likely choose interventions based on the resident's needs, creating selection bias. A resident needing Legal Services may have more complex trauma, making the Legal Services coefficient reflect case complexity rather than intervention quality.

**What we cannot claim:**
- That any specific intervention *causes* emotional improvement. Random assignment of interventions would be needed to establish causality.
- That the model's predictions are accurate for residents at new safehouses with different demographics.

## 8. Deployment Notes

**API endpoint:**
```
POST /api/ml/session-concern-risk
Body: { resident_id, session_type, duration_minutes, interventions_planned, session_date }
Response: { concern_probability, risk_level, recommended_supervisor_availability }
```

**Web app integration:**
- **Process Recording form:** When a social worker logs a planned session, the system shows a concern probability score.
- **Admin Dashboard:** Daily session schedule with color-coded risk flags for supervisors.
- **Process Recording history:** Each past session shows its concern probability vs. actual outcome for model transparency.


In [ ]:
import pickle

with open('p4_counseling_model.pkl', 'wb') as f:
    pickle.dump({'model': best_pipe4, 'features': PRED_FEATURES4}, f)
print("Model saved: p4_counseling_model.pkl")

# Demo inference
sample_session = pd.DataFrame([{
    'session_duration_minutes': 75,
    'session_type_individual': 1,
    'emotion_start_num': 3,  # Sad
    'intv_healing': 1,
    'intv_teaching': 0,
    'intv_legal_services': 0,
    'intv_caring': 1,
    'n_interventions': 2,
    'session_count_prior': 5,
    'initial_risk_num': 3,  # High
    'sub_cat_trafficked': 1,
    'sub_cat_sexual_abuse': 1,
    'has_special_needs': 0,
    'total_incidents': 2,
    'month': 3,
    'day_of_week': 1
}])

prob4 = best_pipe4.predict_proba(sample_session)[0, 1]
print(f"\nSample session: High-risk resident, Sad start, Healing+Caring")
print(f"  Probability of concern being flagged: {prob4:.2%}")
print(f"  Supervisor alert: {'YES' if prob4 >= 0.35 else 'No'}")
